In [23]:
##### Copyright 2020 Google LLC
#
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# QKeras YOLO Object Detection - Quantization for HLS4ml

This notebook demonstrates how to apply __QKeras__ quantization to a YOLO object detection model for FPGA deployment using HLS4ml.

**Workflow:**
1. Build and train a float32 YOLO model using Keras Functional API
2. Apply QKeras quantization with conservative bit-widths (6-8 bits)
3. Fine-tune with Quantization-Aware Training (QAT)
4. Analyze quantization statistics and compare performance
5. Export quantized weights for HLS4ml synthesis

**Dataset:** Bionano cell detection (128x256 rectangular images, 1 class)  
**Model:** TinySIMO YOLO v8 with functional API (QKeras compatible)  
**Image Size:** 128 (height) x 256 (width) - optimized for cell detection  
**Target:** FPGA deployment with ~10-20x efficiency improvement



## Related Work

__QKeras__ has been implemented based on the work of _"B.Moons et al. - Minimum Energy Quantized Neural Networks"_ , Asilomar Conference on Signals, Systems and Computers, 2017 and _“Zhou, S. et al. DoReFa-Net: Training Low Bitwidth Convolutional Neural Networks with Low Bitwidth Gradients,”_ but the framework should be easily extensible. The original code from QNN can be found below.

https://github.com/BertMoons/QuantizedNeuralNetworks-Keras-Tensorflow

__QKeras__ extends QNN by providing a richer set of layers (including SeparableConv2D, DepthwiseConv2D, ternary and stochastic ternary quantizations), besides some functions to aid the estimation for the accumulators and conversion between non-quantized to quantized networks. Finally, our main goal is easy of use, so we attempt to make QKeras layers a true drop-in replacement for Keras, so that users can easily exchange non-quantized layers by quantized ones.

## Layers Implemented in QKeras

The following layers have been implemented in __QKeras__.

- __`QDense`__

- __`QConv1D`__

- __`QConv2D`__

- __`QDepthwiseConv2D`__

- __`QSeparableConv2D`__ (depthwise + pointwise expanded, extended from MobileNet SeparableConv2D implementation)

- __`QActivation`__

- __`QAveragePooling2D`__ (in fact, a AveragePooling2D stacked with a QActivation layer for quantization of the result, so this layer does not exist)

- __`QBatchNormalization`__

- __`QOctaveConv2D`__

It is worth noting that not all functionality is safe at this time to be used with other high-level operations, such as with layer wrappers. For example, `Bidirectional` layer wrappers are used with RNNs.  If this is required, we encourage users to use quantization functions invoked as strings instead of the actual functions as a way through this, but we may change that implementation in the future.

__`QSeparableConv2D`__ is implemented as a depthwise + pointwise quantized expansions, which is extended from the `SeparableConv2D` implementation of MobileNet. With the exception of __`QBatchNormalization`__, if quantizers are not specified, no quantization is applied to the layer and it ends up behaving like the orgininal unquantized layers. On the other hand, __`QBatchNormalization`__ has been implemented differently as if the user does not specify any quantizers as parameters, it uses a set up that has worked best when attempting to implement quantization efficiently in hardware and software, i.e. `gamma` and `variance` with po2 quantizers (as they become shift registers in an implementation, and with further constraining variance po2 quantizer to use quadratic approximation as we take the square root of the variance to obtain the standard deviation), `beta` using po2 quantizer to maintain the dynamic range aspect of the center parameter, and `mean` remaining unquantized, as it inherits the properties of the previous layer.

Activation has been migrated to __`QActivation`__ although it __QKeras__ also recognizes activation parameter used in convolutional and dense layers.

We have improved the setup of quantization as convolution, dense and batch normalization layers now notify the quantizers when the quantizers are used as internal parameters, so the user does not need to worry about setting up options that work best in `weights` and `bias` like `alpha` and `use_stochastic_rounding` (although users may override the automatic setup).

Finally, in the current version, we have eliminated the need to set up the range of the quantizers like `kernel_range` in __`QDense`__. This is automatically computed internally at this point. Although we kept the parameters for backward compatibility, these parameters will be removed in the future.

## Activation Layers and Quantizers Implemented in __QKeras__

Quantizers and activation layers are treated interchangingly in __QKeras__.   

The list of quantizers and its parameters is listed below.

- __`smooth_sigmoid(x)`__

- __`hard_sigmoid(x)`__

- __`binary_sigmoid(x)`__

- __`smooth_tanh(x)`__

- __`hard_tanh(x)`__

- __`binary_tanh(x)`__

- __`quantized_bits(bits=8, integer=0, symmetric=0, keep_negative=1, alpha=None, use_stochastic_rouding=False)(x)`__

- __`bernoulli(alpha=None, temperature=6.0, use_real_sigmoid=True)(x)`__

- __`stochastic_ternary(alpha=None, threshold=None, temperature=8.0, use_real_sigmoid=True)(x)`__

- __`ternary(alpha=None, threshold=None, use_stochastic_rounding=False)(x)`__

- __`stochastic_binary(alpha=None, temperature=6.0, use_real_sigmoid=True)(x)`__

- __`binary(use_01=False, alpha=None, use_stochastic_rounding=False)(x)`__

- __`quantized_relu(bits=8, integer=0, use_sigmoid=0, use_stochastic_rounding=False)(x)`__

- __`quantized_ulaw(bits=8, integer=0, symmetric=0, u=255.0)(x)`__

- __`quantized_tanh(bits=8, integer=0, symmetric=0, use_stochastic_rounding=False)(x)`__

- __`quantized_po2(bits=8, max_value=None, use_stochastic_rounding=False, quadratic_approximation=False)(x)`__

- __`quantized_relu_po2(bits=8, max_value=None, use_stochastic_rounding=False, quadratic_approximation=False)(x)`__

The __`stochastic_*`__ functions and __`bernoulli`__ rely on stochastic versions of the activation functions, so they are best suited for weights and biases.  They draw a random number with uniform distribution from `sigmoid` of the input x, and result is based on the expected value of the activation function. Please refer to the papers if you want to understand the underlying theory, or the documentation in qkeras/quantizers.py. The parameter `temperature` determines how steep the sigmoid function will behave, and the default values seem to work fine.

As we lower the number of bits, rounding becomes problematic as it adds bias to the number system. Numpy attempt to reduce the effects of bias by rounding to even instead of rounding to infinity. Recent results (_"Suyog Gupta, Ankur Agrawal, Kailash Gopalakrishnan, Pritish Narayanan; Deep Learning with Limited Numerical Precision_ [https://arxiv.org/abs/1502.02551]) suggested using stochastic rounding, which uses the fracional part of the number as a probability to round up or down. We can turn on stochastic rounding in some quantizers by setting `use_stochastic_rounding` to `True` in __`quantized_bits`__, __`binary`__, __`ternary`__, __`quantized_relu`__ and __`quantized_tanh`__, __`quantized_po2`__, and __`quantized_relu_po2`__. Please note that if one is considering an efficient hardware or software implementation, we should avoid setting this flag to `True` in activations as it may affect the efficiency of an implementation. In addition, as mentioned before, we already set this flag to `True` in some quantized layers when the quantizers are used as weights/biases.

The parameters `bits` specify the number of bits for the quantization, and `integer` specifies how many bits of `bits` are to the left of the decimal point. Finally, our experience in training networks with __`QSeparableConv2D`__, it is advisable to allocate more bits between the depthwise and the pointwise quantization, and both __`quantized_bits`__ and __`quantized_tanh`__ should use symmetric versions for weights and bias in order to properly converge and eliminate the bias.

We have substantially improved stochastic rounding implementation in __QKeras__ $>= 0.7$, and added a symbolic way to compute alpha in __`binary`__, __`stochastic_binary`__, __`ternary`__, __`stochastic_ternary`__, __`bernoulli`__ and __`quantized_bits`__. Right now, a scale and the threshold (for ternary and stochastic_ternary) can be computed independently of the distribution of the inputs, which is required when using these quantizers in weights.

The main problem in using very small bit widths in large deep learning networks stem from the fact that weights are initialized with variance roughly $\propto \sqrt{1/\tt{fanin}}$, but during the training the variance shifts outwards.  If the smallest quantization representation (threshold in ternary networks) is smaller than $\sqrt{1/\tt{fanin}}$, we run the risk of having the weights stuck at 0 during training. So, the weights need to dynamically adjust to the variance shift from initialization to the final training.  This can be done by scaling the quantization. 

Scale is computed using the formula $\sum(\tt{dot}(Q,x))/\sum(\tt{dot}(Q,Q))$ which is described in several papers, including _Mohammad Rastegari, Vicente Ordonez, Joseph Redmon, Ali Farhadi "XNOR-Net: ImageNet Classification Using Binary Convolutional Neural Networks"_ [https://arxiv.org/abs/1603.05279]. Scale computation is computed for each output channel, making our implementation sometimes behaving like a mini-batch normalization adjustment.  

For __`ternary`__ and __`stochastic_ternary`__, we iterate between scale computation and threshold computation, as presented in _K. Hwang and W. Sung, "Fixed-point feedforward deep neural network design using weights +1, 0, and −1," 2014 IEEE Workshop on Signal Processing Systems (SiPS), Belfast, 2014, pp. 1-6_ which makes the search for threshold and scale tolerant to different input distributions. This is especially important when we need to consider that the threshold shifts depending on the input distribution,  affecting the scale as well, as pointed out by _Fengfu Li, Bo Zhang, Bin Liu, "Ternary Weight Networks"_ [https://arxiv.org/abs/1605.04711]. 

When computing the scale in these quantizers, if `alpha="auto"`, we compute the scale as a floating point number. If `alpha="auto_po2"`, we enforce the scale to be a power of 2, meaning that an actual hardware or software implementation can be performed by just shifting the result of the convolution or dense layer to the right or left by checking the sign of the scale (positive shifts left, negative shifts right), and taking the log2 of the scale.  This behavior is compatible with shared exponent approaches, as it performs a shift adjustment to the channel.

We have implemented a method for each quantizer called __`_set_trainable_parameter`__ that instructs __QKeras__ to set best options when this quantizer is used as a weight or for gamma, variance and beta in __`QBatchNormalization`__, so in principle, users should not worry about this.

The following pictures show the behavior of __`binary`__ vs stochastic rounding in __`binary`__ vs __`stochastic_binary`__ (Figure 1) and __`ternary`__ vs stochastic rounding in __`ternary`__ and __`stochastic_ternary`__ (Figure 2). We generated a normally distributed input with mean 0.0 and standard deviation of 0.02, ordered the data, and ran the quantizer 1,000 times, averaging the result for each case. Note that because of scale, the output does not range from $[-1.0, +1.0]$, but from $[-\tt{scale}, +\tt{scale}]$.


<img src="images/figure1.png" alt="Binary quantizers" title="Figure 1: Behavior of binary quantizers" style="width:60%;height:60%;"/><center>Figure 1: Behavior of binary quantizers</center>

<img src="images/figure2.png" alt="Ternary quantizers" title="Figure 2: Behavior of ternary quantizers" style="width:60%;height:60%;"/><center>Figure 2: Behavior of ternary quantizers</center>


## Using QKeras

__QKeras__ works by tagging all variables and weights/bias created by Keras as well as output of arithmetic layers by quantized functions. Quantized functions can be instantiated directly in __`QDense`__/__`QConv2D`__/__`QSeparableConv2D`__ functions, and they can be passed to __`QActivation`__, which act as a merged quantization and activation function.

In order to successfully quantize a model, users need to replace layers that create variables (trainable or not) (`Dense`, `Conv2D`, etc) by their equivalent ones in __Qkeras__ (__`QDense`__, __`QConv2D`__, etc), and any layers that perform math operations need to be quantized afterwards.

Quantized values are clipped between their maximum and minimum quantized representation (which may be different than $[-1.0, 1.0]$), although for `po2` type of quantizers, we still recommend the users to specify the parameter for `max_value`.

An example of a very simple network is given below in Keras.

In [24]:
# Core imports
import os
import sys
import numpy as np
import tensorflow as tf
import yaml
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

# Add current directory to path
sys.path.insert(0, os.getcwd())

# YOLO model and utilities
from nets.tinysimov35_keras_hls4ml import build_yolo_functional, decode_predictions
from utils.dataset_keras import Dataset
from utils.util_keras import (
    ComputeLoss, EMA, AverageMeter,
    non_max_suppression, compute_ap,
    generate_colors, visualize_predictions
)

# QKeras imports
try:
    from qkeras import *
    from qkeras.utils import model_quantize, quantized_model_debug, model_save_quantized_weights
    from qkeras.estimate import print_qstats
    QKERAS_AVAILABLE = True
    print("✓ QKeras imported successfully")
except ImportError as e:
    print(f"⚠ QKeras import error: {e}")
    print("Install with: pip install qkeras")
    QKERAS_AVAILABLE = False

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print(f"Current directory: {os.getcwd()}")

✓ QKeras imported successfully
TensorFlow version: 2.15.0
GPU Available: []
Current directory: /home/mdi220/simulations/YOLO_Keras


In [25]:
# Verify imports
print("Checking imports...")
print(f"✓ NumPy: {np.__version__}")
print(f"✓ TensorFlow: {tf.__version__}")
print(f"✓ OpenCV: {cv2.__version__}")
print(f"✓ QKeras available: {QKERAS_AVAILABLE}")

# Test YOLO model import with correct rectangular size
try:
    test_model = build_yolo_functional(num_classes=1, img_size=(128, 256))
    print(f"✓ YOLO model builds successfully")
    print(f"  - Input shape: {test_model.input.shape}")
    print(f"  - Output shape: {test_model.output.shape}")
    print(f"  - Stride: {test_model.stride.numpy()}")
    del test_model  # Clean up
except Exception as e:
    print(f"⚠ YOLO model error: {e}")

print("\nAll imports verified!")

Checking imports...
✓ NumPy: 1.26.4
✓ TensorFlow: 2.15.0
✓ OpenCV: 4.11.0
✓ QKeras available: True
✓ YOLO model builds successfully
  - Input shape: (None, 256, 256, 3)
  - Output shape: (None, 32, 32, 65)

All imports verified!


## 1. Configuration and Dataset Setup

Load configuration from YAML and prepare the bionano cell detection dataset.

In [26]:
# Load configuration from YAML
with open('utils/args_bionano.yaml', 'r') as f:
    params = yaml.safe_load(f)

# Configuration - IMPORTANT: img_size is (height, width)
img_size = (128, 256)  # height=128, width=256 (rectangular)
batch_size = 4
num_classes = len(params['names'])  # 1 class: 'cell'
epochs = 50  # Reduced for notebook demo
patience = 20  # Early stopping

print(f"Image size (H x W): {img_size[0]} x {img_size[1]}")
print(f"Batch size: {batch_size}")
print(f"Number of classes: {num_classes}")
print(f"Class names: {params['names']}")

Image size (H x W): 128 x 256
Batch size: 4
Number of classes: 1
Class names: {0: 'cell'}


In [27]:
# Load dataset file lists
train_files = []
with open('Dataset/bionano_cellv2/train.txt') as f:
    for line in f.readlines():
        line = line.rstrip().split('/')[-1]
        train_files.append(os.path.join('Dataset/bionano_cellv2/images/train', line))

val_files = []
with open('Dataset/bionano_cellv2/val.txt') as f:
    for line in f.readlines():
        line = line.rstrip().split('/')[-1]
        val_files.append(os.path.join('Dataset/bionano_cellv2/images/val', line))

print(f"Train: {len(train_files)} images")
print(f"Val: {len(val_files)} images")

# Create dataset objects
train_dataset = Dataset(train_files, img_size, params, augment=True, dtype=tf.float32)
val_dataset = Dataset(val_files, img_size, params, augment=False, dtype=tf.float32)

print(f"\\nDataset created successfully!")
print(f"Sample shape: {train_dataset[0][0].shape}")  # (256, 256, 3)

Train: 172 images
Val: 49 images
\nDataset created successfully!
Sample shape: (128, 256, 3)


## 2. Data Generator

Create a data generator that properly batches samples with batch indexing for YOLO loss computation.

In [28]:
def create_data_generator(dataset, batch_size):
    """Generator that batches samples with proper batch indexing for YOLO loss"""
    def generator():
        batch_samples = []
        batch_targets = []
        
        for i in range(len(dataset)):
            sample, target, shapes = dataset[i]
            
            # Add batch index to targets (first column)
            batch_idx = len(batch_samples)
            if target.shape[0] > 0:
                img_idx = tf.fill([tf.shape(target)[0], 1], tf.cast(batch_idx, target.dtype))
                target = tf.concat([img_idx, target[:, 1:]], axis=1)
            
            batch_samples.append(sample)
            if target.shape[0] > 0:
                batch_targets.append(target)
            
            # Yield complete batch
            if (i + 1) % batch_size == 0:
                stacked_samples = tf.stack(batch_samples)
                stacked_targets = tf.concat(batch_targets, axis=0) if batch_targets else tf.zeros((0, 6), dtype=tf.float32)
                
                yield stacked_samples, stacked_targets
                
                batch_samples = []
                batch_targets = []
    
    return tf.data.Dataset.from_generator(
        generator,
        output_types=(tf.float32, tf.float32),
        output_shapes=((batch_size, img_size[0], img_size[1], 3), (None, 6))
    ).prefetch(tf.data.AUTOTUNE)

print("Data generator function defined")

Data generator function defined


## 3. Build Float32 YOLO Model

Create the functional YOLO model (required for QKeras compatibility).

In [ ]:
# Build float32 baseline model
model = build_yolo_functional(
    num_classes=num_classes,
    img_size=img_size,
    dtype=tf.float32
)

print("Model built successfully!")
print(f"Input shape: (None, {img_size[0]}, {img_size[1]}, 3)")
print(f"Output shape: {model.output.shape}")
print(f"Stride: {model.stride}")
print(f"DFL channels: {model.dfl_ch}")
print(f"Number of classes: {model.nc}")

model.summary()

Model built successfully!
Input shape: (None, 128, 256, 3)
Output shape: (None, 16, 32, 65)
Stride: [8.]
DFL channels: 16
Number of classes: 1
Model: "yolo_functional"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input (InputLayer)          [(None, 128, 256, 3)]        0         []                            
                                                                                                  
 backbone_conv1 (Conv2D)     (None, 64, 128, 4)           108       ['input[0][0]']               
                                                                                                  
 backbone_bn1 (BatchNormali  (None, 64, 128, 4)           16        ['backbone_conv1[0][0]']      
 zation)                                                                                          
                                        

In [ ]:
# Setup loss function and optimizer
criterion = ComputeLoss(model, params, dtype=tf.float32)

# SGD with momentum (YOLO standard)
learning_rate = params['lr0']  # 0.01
momentum = params['momentum']   # 0.937

optimizer = tf.keras.optimizers.SGD(
    learning_rate=learning_rate,
    momentum=momentum,
    nesterov=True
)

print(f"Loss function: ComputeLoss")
print(f"Optimizer: SGD with momentum={momentum}, lr={learning_rate}")

Loss function: ComputeLoss
Optimizer: SGD with momentum=0.937, lr=0.01


## 4. Evaluation Function

Define evaluation function to compute mAP metrics.

In [ ]:
def evaluate_model(model, dataset, params, max_samples=50):
    """Evaluate model on validation set and compute mAP metrics"""
    metrics = []
    _, h, w, _ = model.input.shape
    
    for i in range(min(len(dataset), max_samples)):
        sample, target, shapes = dataset[i]
        sample = tf.expand_dims(sample, 0)  # Add batch dim
        
        # Forward pass
        outputs = model(sample, training=False)
        
        # Decode predictions (functional model outputs raw features)
        outputs = decode_predictions(outputs, model.stride, model.nc, model.dfl_ch, dtype=tf.float32)
        
        # NMS
        detections = non_max_suppression(outputs, conf_threshold=0.25, iou_threshold=0.45)
        pred = detections[0]
        
        # Process ground truth - scale to pixel coordinates
        if target.shape[0] > 0:
            gt = target.numpy()
            # Scale GT from normalized to pixel coords
            scale_tensor = np.array([1.0, 1.0, float(w), float(h), float(w), float(h)])
            gt = gt * scale_tensor
            
            # Convert to numpy for metric calculation
            if pred is not None and len(pred) > 0:
                pred_np = pred.numpy() if hasattr(pred, 'numpy') else pred
                
                # Calculate IoU-based metrics (simplified)
                # Convert GT xywh -> xyxy
                label_boxes = np.zeros((len(gt), 5), dtype=gt.dtype)
                label_boxes[:, 0] = gt[:, 1]  # cls
                label_boxes[:, 1] = gt[:, 2] - gt[:, 4] / 2.0  # x1
                label_boxes[:, 2] = gt[:, 3] - gt[:, 5] / 2.0  # y1
                label_boxes[:, 3] = gt[:, 2] + gt[:, 4] / 2.0  # x2
                label_boxes[:, 4] = gt[:, 3] + gt[:, 5] / 2.0  # y2
                
                # IoU thresholds for mAP@0.5:0.95
                iou_v = np.linspace(0.5, 0.95, 10)
                correct = np.zeros((pred_np.shape[0], len(iou_v)), dtype=bool)
                
                # Simple IoU matching
                for j, iou_threshold in enumerate(iou_v):
                    for det_idx in range(pred_np.shape[0]):
                        det_box = pred_np[det_idx, :4]
                        det_class = pred_np[det_idx, 5]
                        
                        for gt_idx in range(len(label_boxes)):
                            gt_box = label_boxes[gt_idx, 1:5]
                            gt_class = label_boxes[gt_idx, 0]
                            
                            if int(round(det_class)) == int(round(gt_class)):
                                # Calculate IoU
                                x1 = max(det_box[0], gt_box[0])
                                y1 = max(det_box[1], gt_box[1])
                                x2 = min(det_box[2], gt_box[2])
                                y2 = min(det_box[3], gt_box[3])
                                
                                if x2 > x1 and y2 > y1:
                                    intersection = (x2 - x1) * (y2 - y1)
                                    det_area = (det_box[2] - det_box[0]) * (det_box[3] - det_box[1])
                                    gt_area = (gt_box[2] - gt_box[0]) * (gt_box[3] - gt_box[1])
                                    union = det_area + gt_area - intersection
                                    iou = intersection / union if union > 0 else 0
                                    
                                    if iou >= iou_threshold:
                                        correct[det_idx, j] = True
                                        break
                
                conf = pred_np[:, 4]
                pred_cls = pred_np[:, 5]
                true_cls = label_boxes[:, 0]
                metrics.append((correct, conf, pred_cls, true_cls))
            elif len(gt) > 0:
                # No detections but have GT
                metrics.append((np.zeros((0, 10), dtype=bool), np.zeros(0), np.zeros(0), gt[:, 1]))
    
    # Compute mAP
    if len(metrics) > 0:
        metrics = [np.concatenate(x, 0) for x in zip(*metrics)]
        _, _, _, _, map50, mean_ap = compute_ap(*metrics)
    else:
        map50 = mean_ap = 0
    
    return map50, mean_ap

print("Evaluation function defined")

## 5. Training Loop

Train the float32 baseline model with gradient clipping and early stopping.

In [33]:
# Create data loader
train_loader = create_data_generator(train_dataset, batch_size)

# Training state
best_map = 0
patience_counter = 0
history = {'loss': [], 'val_map50': [], 'val_map': []}

print("Starting training...")
print(f"Epochs: {epochs}, Batch size: {batch_size}, Patience: {patience}")

for epoch in range(epochs):
    epoch_loss = AverageMeter()
    
    # Turn off mosaic for last 10 epochs
    if epochs - epoch == 10:
        train_dataset.mosaic = False
        print(f"\nTurning off mosaic augmentation at epoch {epoch+1}")
    
    # Training
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
    for samples, targets in pbar:
        with tf.GradientTape() as tape:
            outputs = model(samples, training=True)
            loss = criterion(outputs, targets)
            loss *= batch_size  # Scale loss
        
        # Backward pass with gradient clipping
        gradients = tape.gradient(loss, model.trainable_variables)
        gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        
        # FIX: Convert loss to scalar float
        loss_val = float(loss.numpy())
        epoch_loss.update(loss_val, samples.shape[0])
        
        # FIX: Convert avg to float for formatting
        pbar.set_postfix({'loss': f'{float(epoch_loss.avg):.4f}'})
    
    print(f"Epoch {epoch+1}/{epochs} - Loss: {float(epoch_loss.avg):.4f}")
    history['loss'].append(float(epoch_loss.avg))
    
    # Validation every 5 epochs
    if (epoch + 1) % 5 == 0:
        print("Evaluating on validation set...")
        val_map50, val_map = evaluate_model(model, val_dataset, params, max_samples=50)
        history['val_map50'].append(val_map50)
        history['val_map'].append(val_map)
        
        print(f"Validation - mAP@50: {val_map50:.3f}, mAP@50:95: {val_map:.3f}")
        
        # Save best model
        if val_map > best_map:
            best_map = val_map
            patience_counter = 0
            model.save_weights('best_float32.h5')
            print(f"✓ New best model saved! (mAP: {best_map:.3f})")
        else:
            patience_counter += 1
            print(f"No improvement ({patience_counter}/{patience})")
        
        # Early stopping
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\nTraining completed! Best mAP: {best_map:.3f}")


Starting training...
Epochs: 50, Batch size: 4, Patience: 20


Epoch 1/50: 0it [00:00, ?it/s]

Epoch 1/50: 43it [00:08,  4.80it/s, loss=14.0206]


Epoch 1/50 - Loss: 14.0206


Epoch 2/50: 43it [00:08,  4.87it/s, loss=13.6667]


...

Epoch 48/50 - Loss: 12.3586


Epoch 49/50: 43it [00:08,  4.98it/s, loss=12.0570]


Epoch 49/50 - Loss: 12.0570


Epoch 50/50: 43it [00:08,  4.93it/s, loss=11.7961]


Epoch 50/50 - Loss: 11.7961
Evaluating on validation set...
Validation - mAP@50: 0.995, mAP@50:95: 0.742
No improvement (3/20)

Training completed! Best mAP: 0.788


## 6. Quantization Configuration

Apply QKeras quantization with conservative bit-widths (6-8 bits) for YOLO detection.

In [34]:
# Define quantization configuration (conservative for YOLO)
quantization_config = {
    # Backbone convolutions - 6 bits for feature extraction
    "backbone_conv1": {
        "kernel_quantizer": "quantized_bits(6,0,1,alpha='auto')",
        "bias_quantizer": "quantized_bits(6)"
    },
    "backbone_conv2": {
        "kernel_quantizer": "quantized_bits(6,0,1,alpha='auto')",
        "bias_quantizer": "quantized_bits(6)"
    },
    "backbone_conv3": {
        "kernel_quantizer": "quantized_bits(6,0,1,alpha='auto')",
        "bias_quantizer": "quantized_bits(6)"
    },
    "backbone_conv4": {
        "kernel_quantizer": "quantized_bits(6,0,1,alpha='auto')",
        "bias_quantizer": "quantized_bits(6)"
    },
    
    # Batch normalization - power-of-2 for hardware efficiency
    "QBatchNormalization": {
        "gamma_quantizer": "quantized_po2(4)",
        "beta_quantizer": "quantized_po2(4)"
    },
    
    # Detection heads - 8 bits for critical box/class predictions
    "box_conv": {
        "kernel_quantizer": "quantized_bits(8,0,1,alpha='auto')",
        "bias_quantizer": "quantized_bits(8)"
    },
    "cls_conv": {
        "kernel_quantizer": "quantized_bits(8,0,1,alpha='auto')",
        "bias_quantizer": "quantized_bits(8)"
    },
    
    # Activations - 6 bits with 2 integer bits
    "QActivation": {
        "relu": "quantized_relu(6,2)"
    }
}

print("Quantization configuration defined:")
print("- Backbone: 6-bit weights")
print("- Detection heads: 8-bit weights")
print("- Batch norm: 4-bit power-of-2")
print("- Activations: 6-bit ReLU")

Quantization configuration defined:
- Backbone: 6-bit weights
- Detection heads: 8-bit weights
- Batch norm: 4-bit power-of-2
- Activations: 6-bit ReLU


In [49]:
# QUANTIZATION: Build quantized functional model and transfer weights
# This uses the same functional API approach as the float32 model

# Load best float32 model
model.load_weights('best_float32.h5')
print("Loaded best float32 model weights")

# Build quantized model using functional API (same structure as float32 model)
print("\nBuilding quantized model with functional API...")
from nets.tinysimov35_keras_quantized_functional import yolo_v8_s_quantized_functional

qmodel = yolo_v8_s_quantized_functional(
    num_classes=num_classes,
    img_size=img_size,
    weight_bits=8,
    activation_bits=8,
    dtype=tf.float32
)

print("✓ Quantized model built")

# Transfer weights layer by layer (both models have same structure)
print("\nTransferring weights from float32 model...")

# Map layers by name
float_layer_dict = {layer.name: layer for layer in model.layers}
transferred = 0
skipped = 0

for qlayer in qmodel.layers:
    # Map quantized layer names to float32 layer names
    # backbone_qconv1 -> backbone_conv1, etc.
    float_name = qlayer.name.replace('qconv', 'conv').replace('qrelu', 'relu')
    
    if float_name in float_layer_dict:
        flayer = float_layer_dict[float_name]
        
        # Transfer weights if layer has them
        try:
            float_weights = flayer.get_weights()
            if len(float_weights) > 0:
                qlayer.set_weights(float_weights)
                print(f"  ✓ {flayer.name} -> {qlayer.name}")
                transferred += 1
        except Exception as e:
            print(f"  ⚠ Failed {flayer.name} -> {qlayer.name}: {e}")
            skipped += 1

print(f"\n✓ Weight transfer complete: {transferred} layers transferred, {skipped} skipped")

print("\n" + "="*80)
print("QUANTIZED MODEL SUMMARY")
print("="*80)
qmodel.summary()

print("\n" + "="*80)
print("QUANTIZER INFORMATION")
print("="*80)

# Check for quantized layers
from qkeras import QConv2D, QActivation
has_quantizers = False
qconv_count = 0
qact_count = 0

for layer in qmodel.layers:
    if isinstance(layer, QConv2D):
        qconv_count += 1
        kernel_q = str(layer.kernel_quantizer_internal) if hasattr(layer, 'kernel_quantizer_internal') else str(layer.kernel_quantizer)
        bias_q = str(layer.bias_quantizer_internal) if hasattr(layer, 'bias_quantizer_internal') else str(layer.bias_quantizer)
        print(f"{layer.name:30s} - kernel: {kernel_q}")
        if layer.bias_quantizer is not None:
            print(f"{'':30s}   bias:   {bias_q}")
        has_quantizers = True
    elif isinstance(layer, QActivation):
        qact_count += 1
        print(f"{layer.name:30s} - activation: {str(layer.quantizer)}")
        has_quantizers = True

if not has_quantizers:
    print("⚠ WARNING: No quantizers found! Model is not quantized.")
else:
    print(f"\n✓ Quantization active:")
    print(f"  - {qconv_count} QConv2D layers")
    print(f"  - {qact_count} QActivation layers")
    
print("="*80)


Loaded best float32 model weights

Building quantized model with functional API...
✓ Quantized model built

Transferring weights from float32 model...
  ✓ backbone_conv1 -> backbone_qconv1
  ✓ backbone_bn1 -> backbone_bn1
  ✓ backbone_conv2 -> backbone_qconv2
  ✓ backbone_bn2 -> backbone_bn2
  ✓ backbone_conv3 -> backbone_qconv3
  ✓ backbone_bn3 -> backbone_bn3
  ✓ backbone_conv4 -> backbone_qconv4
  ✓ backbone_bn4 -> backbone_bn4
  ✓ box_conv -> box_qconv
  ✓ cls_conv -> cls_qconv

✓ Weight transfer complete: 10 layers transferred, 0 skipped

QUANTIZED MODEL SUMMARY
Model: "yolo_quantized_functional"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input (InputLayer)          [(None, 128, 256, 3)]        0         []                            
                                                                                              

## 7. Quantization-Aware Training

Fine-tune the quantized model to recover accuracy. Run the cell below to perform QAT for 20 epochs.

In [50]:
# REPLACE CELL 37 WITH THIS CODE
# This is the QAT (Quantization-Aware Training) cell

# QAT Training
print("\n## Starting Quantization-Aware Training ##\n")
qat_epochs = 20
qat_lr = 0.001
best_qmap = 0

qat_optimizer = tf.keras.optimizers.SGD(learning_rate=qat_lr, momentum=momentum, nesterov=True)
qcriterion = ComputeLoss(qmodel, params, dtype=tf.float32)

for epoch in range(qat_epochs):
    epoch_loss = AverageMeter()
    train_loader_qat = create_data_generator(train_dataset, batch_size)
    
    pbar = tqdm(train_loader_qat, desc=f'QAT {epoch+1}/{qat_epochs}')
    for samples, targets in pbar:
        with tf.GradientTape() as tape:
            outputs = qmodel(samples, training=True)
            loss = qcriterion(outputs, targets) * batch_size
        
        gradients = tape.gradient(loss, qmodel.trainable_variables)
        gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
        qat_optimizer.apply_gradients(zip(gradients, qmodel.trainable_variables))
        
        # Convert to float to avoid formatting error
        loss_val = float(loss.numpy())
        epoch_loss.update(loss_val, samples.shape[0])
        pbar.set_postfix({'loss': f'{float(epoch_loss.avg):.4f}'})
    
    print(f"Epoch {epoch+1} - Loss: {float(epoch_loss.avg):.4f}")
    
    if (epoch + 1) % 5 == 0:
        val_map50, val_map = evaluate_model(qmodel, val_dataset, params, max_samples=50)
        print(f"mAP@50: {val_map50:.3f}, mAP: {val_map:.3f}")
        if val_map > best_qmap:
            best_qmap = val_map
            qmodel.save_weights('best_quantized.h5')
            print("✓ Best quantized model saved!")

print(f"\nQAT completed! Best mAP: {best_qmap:.3f}")



## Starting Quantization-Aware Training ##



QAT 1/20: 0it [00:00, ?it/s]

QAT 1/20: 43it [00:11,  3.82it/s, loss=13.9909]


Epoch 1 - Loss: 13.9909


QAT 2/20: 43it [00:11,  3.77it/s, loss=12.9724]


...

Epoch 18 - Loss: 11.8865


QAT 19/20: 43it [00:10,  3.93it/s, loss=12.0577]


Epoch 19 - Loss: 12.0577


QAT 20/20: 43it [00:11,  3.85it/s, loss=11.9609]


Epoch 20 - Loss: 11.9609
mAP@50: 0.990, mAP: 0.746

QAT completed! Best mAP: 0.791


Great! it is relatively easy to create a network that converges in MNIST with very high test accuracy. The reader should note that we named all the layers as it will make it easier to automatically convert the network by name.

In [51]:
# FIXED VISUALIZATION CODE - Shows both GT and predictions correctly

# Setup
results_dir = 'qkeras_results'
os.makedirs(results_dir, exist_ok=True)
class_colors = generate_colors(num_classes)

print("Generating visualizations with bounding boxes...")
print("(Ground truth on left, predictions on right)\n")

for idx in range(10):
    # Get sample
    sample, target, shapes = val_dataset[idx]
    sample_batch = tf.expand_dims(sample, 0)
    
    # Get predictions
    outputs = qmodel(sample_batch, training=False)
    outputs_decoded = decode_predictions(outputs, qmodel.stride, qmodel.nc, qmodel.dfl_ch)
    detections = non_max_suppression(outputs_decoded, 0.25, 0.45)
    
    # Prepare GT boxes in correct format for visualization
    # Target format from dataset: [batch_idx, cls, x_norm, y_norm, w_norm, h_norm]
    # Need to convert to: [img_idx, cls, x_pixel, y_pixel, w_pixel, h_pixel]
    
    _, h, w, _ = sample_batch.shape
    
    if target.shape[0] > 0:
        # Scale normalized coords to pixel coords
        gt_boxes_viz = target.numpy().copy()
        
        # Add batch index column if not present
        if gt_boxes_viz.shape[1] == 5:
            # Format: [cls, x, y, w, h] -> add img_idx
            batch_col = np.zeros((gt_boxes_viz.shape[0], 1))
            gt_boxes_viz = np.concatenate([batch_col, gt_boxes_viz], axis=1)
        
        # Scale from normalized to pixel coordinates
        # Columns: [img_idx, cls, x, y, w, h]
        gt_boxes_viz[:, 2] *= w  # x_center
        gt_boxes_viz[:, 3] *= h  # y_center
        gt_boxes_viz[:, 4] *= w  # width
        gt_boxes_viz[:, 5] *= h  # height
        
        gt_boxes_tensor = tf.constant(gt_boxes_viz, dtype=tf.float32)
    else:
        gt_boxes_tensor = tf.zeros((0, 6), dtype=tf.float32)
    
    # Visualize
    visualize_predictions(
        sample_batch, 
        detections, 
        gt_boxes_tensor,  # Pass scaled GT boxes
        tf.expand_dims(shapes, 0), 
        params,
        class_colors, 
        results_dir, 
        idx
    )
    
    # Print info
    pred = detections[0]
    n_det = len(pred) if pred is not None else 0
    n_gt = target.shape[0]
    print(f"  Image {idx+1:2d}: {n_gt} GT boxes, {n_det} predictions")

print(f"\n✓ Visualizations saved to: {results_dir}/")
print("  Files: result_0.png through result_9.png")
print("  Format: Ground Truth (left) | Predictions (right)")


Generating visualizations with bounding boxes...
(Ground truth on left, predictions on right)

  Image  1: 1 GT boxes, 1 predictions
  Image  2: 1 GT boxes, 1 predictions
  Image  3: 1 GT boxes, 1 predictions
  Image  4: 1 GT boxes, 1 predictions
  Image  5: 1 GT boxes, 0 predictions
  Image  6: 1 GT boxes, 1 predictions
  Image  7: 1 GT boxes, 1 predictions
  Image  8: 1 GT boxes, 1 predictions
  Image  9: 1 GT boxes, 1 predictions
  Image 10: 1 GT boxes, 1 predictions

✓ Visualizations saved to: qkeras_results/
  Files: result_0.png through result_9.png
  Format: Ground Truth (left) | Predictions (right)


In [39]:
model.summary()

Model: "yolo_functional"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input (InputLayer)          [(None, 128, 256, 3)]        0         []                            
                                                                                                  
 backbone_conv1 (Conv2D)     (None, 64, 128, 4)           108       ['input[0][0]']               
                                                                                                  
 backbone_bn1 (BatchNormali  (None, 64, 128, 4)           16        ['backbone_conv1[0][0]']      
 zation)                                                                                          
                                                                                                  
 backbone_relu1 (ReLU)       (None, 64, 128, 4)           0         ['backbone_bn1[0

You should note that we had to lower the learning rate and train the network for longer time. On the other hand, the network should not involve in any multiplications in the convolution layers, and very small multipliers in the dense layers.

Please note that the last `Activation` was not changed to __`QActivation`__ as during inference we usually perform the operation `argmax` on the result instead of `softmax`.

It seems it is a lot of code to write besides the main network, but in fact, this additional code is only specifying the sizes of the weights and the sizes of the outputs in the case of the activations.  Right now, we do not have a way to extract this information from the network structure or problem we are trying to solve, and if we quantize too much a layer, we may end up not been able to recover from that later on.

## Converting a Model Automatically

In addition to the drop-in replacement of Keras functions, we have written the following function to assist anyone who wants to quantize a network.

__`model_quantize(model, quantizer_config, activation_bits, custom_objects=None, transfer_weights=False)`__

This function converts an non-quantized model (such as the one from `model` in the previous example) into a quantized version, by applying a configuration specified by the dictionary `quantizer_config`, and `activation_bits` specified for unamed activation functions, with this parameter probably being removed in future versions.

The parameter `custom_objects` specifies object dictionary unknown to Keras, required when you copy a model with lambda layers, or customized layer functions, for example, and if `transfer_weights` is `True`, the returned model will have as initial weights the weights from the original model, instead of using random initial weights.

The dictionary specified in `quantizer_config` can be indexed by a layer name or layer class name. In the example below, conv2d_1 corresponds to the first convolutional layer of the example, while  QConv2D corresponds to the default behavior of two dimensional convolutional layers. The reader should note that right now we recommend using __`QActivation`__ with a dictionary to avoid the conversion of activations such as `softmax` and `linear`.  In addition, although we could use `activation` field in the layers, we do not recommend that. 

`{
  "conv2d_1": {
      "kernel_quantizer": "stochastic_ternary",
      "bias_quantizer": "quantized_po2(4)"
  },
  "QConv2D": {
      "kernel_quantizer": "stochastic_ternary",
      "bias_quantizer": "quantized_po2(4)"
  },
  "QDense": {
      "kernel_quantizer": "quantized_bits(3,0,1)",
      "bias_quantizer": "quantized_bits(3)"
  },
  "act_1": "quantized_relu(2)",
  "QActivation": { "relu": "quantized_relu(2)" }
}`

In the following example, we will quantize the model using a different strategy.


In [ ]:
config = {
  "conv2d_1": {
      "kernel_quantizer": "stochastic_binary",
      "bias_quantizer": "quantized_po2(4)"
  },
  "QConv2D": {
      "kernel_quantizer": "stochastic_ternary",
      "bias_quantizer": "quantized_po2(4)"
  },
  "QDense": {
      "kernel_quantizer": "quantized_bits(4,0,1)",
      "bias_quantizer": "quantized_bits(4)"
  },
  "QActivation": { "relu": "binary" },
  "act_2": "quantized_relu(3)",
}

In [ ]:
from qkeras.utils import model_quantize

qmodel = model_quantize(model, config, 4, transfer_weights=True)

for layer in qmodel.layers:
    if hasattr(layer, "kernel_quantizer"):
        print(layer.name, "kernel:", str(layer.kernel_quantizer_internal), "bias:", str(layer.bias_quantizer_internal))
    elif hasattr(layer, "quantizer"):
        print(layer.name, "quantizer:", str(layer.quantizer))

print()
qmodel.summary()

TypeError: Could not locate class 'QConv2D'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'module': 'keras.layers', 'class_name': 'QConv2D', 'config': {'name': 'backbone_conv1', 'trainable': True, 'dtype': 'float32', 'filters': 4, 'kernel_size': [3, 3], 'strides': [2, 2], 'padding': 'same', 'data_format': 'channels_last', 'dilation_rate': [1, 1], 'groups': 1, 'activation': 'linear', 'use_bias': False, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_regularizer': None, 'activity_regularizer': None, 'kernel_constraint': None, 'bias_constraint': None, 'kernel_quantizer': 'stochastic_ternary', 'bias_quantizer': None}, 'registered_name': None, 'build_config': {'input_shape': [None, 128, 25

In [ ]:
qmodel.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(0.001),
    metrics=["accuracy"])

In [ ]:
qmodel.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test), verbose=True)

in addition to __`model_quantize`__, __QKeras__ offers the additional utility functions.

__`BinaryToThermometer(x, classes, value_range, with_residue=False, merge_with_channels, use_two_hot_encoding=False)`__

This function converts a dense binary encoding of inputs to one-hot (with scales).

Given input matrix `x` with values (for example) 0, 1, 2, 3, 4, 5, 6, 7, create a number of classes as follows:

If classes=2, value_range=8, with_residue=0, a true one-hot representation is created, and the remaining bits are truncated, using one bit representation.

`
0 - [1,0] 1 - [1,0] 2 - [1,0] 3 - [1,0]
4 - [0,1] 5 - [0,1] 6 - [0,1] 7 - [0,1]
`

If classes=2, value_range=8, with_residue=1, the residue is added to the one-hot class, and the class will use 2 bits (for the remainder) + 1 bit (for the one hot)

`
0 - [1,0] 1 - [1.25,0] 2 - [1.5,0] 3 - [1.75,0]
4 - [0,1] 5 - [0,1.25] 6 - [0,1.5] 7 - [0,1.75]
`

The arguments of this functions are as follows:

`
x: the input vector we want to convert. typically its dimension will be
      (B,H,W,C) for an image, or (B,T,C) or (B,C) for for a 1D signal, where
      B=batch, H=height, W=width, C=channels or features, T=time for time
      series.
classes: the number of classes to (or log2(classes) bits) to use of the
      values.
value_range: max(x) - min(x) over all possible x values (e.g. for 8 bits,
      we would use 256 here).
with_residue: if true, we split the value range into two sets and add
      the decimal fraction of the set to the one-hot representation for partial
      thermometer representation.
merge_with_channels: if True, we will not create a separate dimension
      for the resulting matrix, but we will merge this dimension with
      the last dimension.
use_two_hot_encoding: if true, we will distribute the weight between
      the current value and the next one to make sure the numbers will always
      be < 1.
`

__`model_save_quantized_weights(model, filename)`__

This function saves the quantized weights in the model or writes the quantized weights in the file `filename` for production, as the weights during training are maintained non-quantized because of training. Typically, you should call this function before productizing the final model.  The saved model is compatible with Keras for inference, so for power-of-2 quantization, we will not return `(sign, round(log2(weights)))`, but rather `(-1)**sign*2**(round(log2(weights)))`. We also return a dictionary containing the name of the layer and the quantized weights, and for power-of-2 quantizations, we will return `sign` and `round(log2(weights))` so that other tools can properly process that.

__`load_qmodel(filepath, custom_objects=None, compile=True)`__

Load quantized model from Keras's model.save() h5 file, where filepath is the path to the filename, custom_objects is an optional dictionary mapping names (strings) to custom classes or functions to be considered during deserialization, and compile instructs __QKeras__ to compile the model after reading it.  If an optimizer was found as part of the saved model, the model is already compiled. Otherwise, the model is uncompiled and a warning will be displayed. When compile is set to `False`, the compilation is omitted without any warning.

__`print_model_sparsity(model)`__

Prints sparsity for the pruned layers in the model.

__`quantized_model_debug(model, X_test, plot=False)`__

Debugs and plots model weights and activations. It is usually useful to print weights, biases and activations for inputs and outputs when debugging a model.  model contains the mixed quantized/unquantized layers for a model. We only print/plot activations and weights/biases for quantized models with the exception of Activation. X_test is the set of inputs we will use to compute activations, and we recommend that the user uses a subsample from the entire set he/she wants to debug. if plot is True, we also plot weights and activations (inputs/outputs) for each layer.

__`extract_model_operations(model)`__

As each operation depends on the quantization method for the weights/bias and on the quantization of the inputs, we estimate which operations are required for each layer of the quantized model.  For example, inputs of a __`QDense`__ layer are quantized using __`quantized_relu_po2`__ and weights are quantized using __`quantized_bits`__, the matrix multiplication can be implemented as a barrel shifter + accumulator without multiplication operations. Right now, we return for each layer one of the following operations: `mult`, `barrel`, `mux`, `adder`, `xor`, and the sizes of the operator.

We are currently refactoring this function and it may be substantially changed in the future.

__`print_qstats(model)`__

Prints statistics of number of operations per operation type and layer so that user can see how big the model is. This function utilizes __`extract_model_operations`__.

An example of the output is presented below.

`Number of operations in model:
    conv2d_0_m                    : 25088 (smult_4_8)
    conv2d_1_m                    : 663552 (smult_4_4)
    conv2d_2_m                    : 147456 (smult_4_4)
    dense                         : 5760  (smult_4_4)

Number of operation types in model:
    smult_4_4                     : 816768
    smult_4_8                     : 25088`

In this example, smult_4_4 stands for 4x4 bit signed multiplication and smult_4_8 stands for 8x4 signed multiplication.

We are currently refactoring this function and it may be substantially changed in the future.


In the quantized network `qmodel`, let's print the statistics of the model and weights.

In [ ]:
print_qstats(qmodel)

In [ ]:
from qkeras.utils import quantized_model_debug

quantized_model_debug(qmodel, x_test, plot=False)

Where the values in `conv2d_1 -4.6218   4.0295 ( -1.0000   1.0000) ( -0.5000   0.5000) a(  0.125000   0.500000)` corresponde to min and max values of the output of the convolution layer, weight ranges (min and max), bias (min and max) and alpha (min and max).